In [ ]:
import os
import numpy as np
from scipy.sparse import csc_array
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
import seaborn as sns

In [ ]:
theta_0 = 0.0 * np.pi
omega_0 = 0.0

g = 9.81
nu = 0.01
a = 0.8
omega = 3.0
l = 1.0
tol = 1e-8


def pivot(*, t):
    return a * np.cos(omega * t)


def forcing(*, t):
    return -a * omega**2 * np.cos(omega * t)


def residual(*, t, theta, theta_0=theta_0, omega_0=omega_0):
    assert len(t) == len(theta)

    N = len(theta) - 1
    r = np.zeros_like(theta)

    a = (theta[0 : N - 1] - 2 * theta[1:N] + theta[2 : N + 1]) / h**2
    v = (theta[2 : N + 1] - theta[0 : N - 1]) / (2 * h)
    f = forcing(t=t)
    a_0 = g * np.sin(theta_0) + nu * omega_0 + f[0] * np.cos(theta_0)

    r[0] = theta[0] - theta_0
    r[1] = l * (theta[1] - theta[0]) / h - l * omega_0 + 0.5 * h * a_0
    r[2 : N + 1] = l * a + nu * v + g * np.sin(theta[1:N]) + f[1:N] * np.cos(theta[1:N])

    return r


def jacobian(*, t, theta):
    # dr / dtheta

    assert len(t) == len(theta)

    N = len(theta) - 1
    f = forcing(t=t)

    # J[0, 0] = 1.0
    # J[1, 0] = -l / h
    # J[1, 1] = l / h
    # J[2, 0] = 1.0 / h**2
    # J[2, 1] = -2.0 / h**2 + g / l * np.cos(theta[1])
    # J[2, 2] = 1.0 / h**2
    # etc.

    r = np.arange(2, N + 1)
    row_idx = np.concatenate([[0, 1, 1], np.repeat(r, 3)])
    col_idx = np.concatenate([[0, 0, 1], np.stack([r - 2, r - 1, r]).T.flatten()])
    values = np.zeros(3 * N)
    values[:3] = [1.0, -l / h, l / h]

    for i in range(2, N + 1):

        values[3 * i - 3] = l / h**2 - nu / (2 * h)
        values[3 * i - 2] = (
            -2.0 * l / h**2 + g * np.cos(theta[i - 1]) - f[i - 1] * np.sin(theta[i - 1])
        )
        values[3 * i - 1] = l / h**2 + nu / (2 * h)

    return csc_array((values, (row_idx, col_idx)), shape=(N + 1, N + 1))


def newton_raphson(*, t, theta_init=None, theta_0=theta_0, omega_0=omega_0):
    if theta_init is None:
        theta = np.zeros_like(t)
    else:
        theta = theta_init

    assert len(t) == len(theta)

    for it in range(100):
        r = residual(t=t, theta=theta, theta_0=theta_0, omega_0=omega_0)

        # print(np.linalg.norm(r))

        if np.linalg.norm(r) < tol * N:
            break

        # theta += np.linalg.solve(jacobian(theta, t=t), -r)
        theta += spsolve(jacobian(t=t, theta=theta), -r)
    else:
        raise RuntimeError("NOT CONVERGED!")

    return theta

In [ ]:
T = 45
hs = [0.200, 0.100, 0.050, 0.020, 0.010, 0.005, 0.002, 0.001]
ts = []
thetas = []

for h in hs:
    # h = 0.001
    N = int(T / h)
    t = np.linspace(0, T, N + 1)

    print(h)
    N_chunk = int(1 / h)

    t_chunks = []
    theta_chunks = []
    theta_full = np.zeros(N + 1)

    for i in range(int(T)):
        if len(theta_chunks) == 0:
            theta_init = theta_0
            omega_init = omega_0
        else:
            theta_init = theta_chunks[-1][-1]
            omega_init = (theta_chunks[-1][-1] - theta_chunks[-1][-2]) / h

        idx = (i * N_chunk, (i + 1) * N_chunk + 1)
        t_chunk = t[idx[0] : idx[1]]
        theta_chunk = newton_raphson(t=t_chunk, theta_0=theta_init, omega_0=omega_init)

        t_chunks.append(t_chunk)
        theta_chunks.append(theta_chunk)
        theta_full[idx[0] : idx[1]] = theta_chunk

    theta = newton_raphson(t=t, theta_init=theta_full)

    plt.figure()
    for t_chunk, theta_chunk in zip(t_chunks, theta_chunks):
        plt.plot(t_chunk, theta_chunk)
    plt.plot(t, theta)
    plt.show()

    ts.append(t)
    thetas.append(theta)

In [ ]:
colors = dict(zip(hs, sns.color_palette("rocket_r", n_colors=len(hs))))

fig, ax = plt.subplots(figsize=(5, 3), tight_layout=True)

for h, t, theta in zip(hs, ts, thetas):
    color = colors[h]
    ax.plot(t, theta, c=color, label=h)

# ax.set_xlim(lims_E)
# ax.set_ylim(lims_k)
ax.set_yticks(
    [4 * i * np.pi for i in range(6)],
    ["0"] + [r"${}\pi$".format(4 * i) for i in range(1, 6)],
)
# ax.set_yticks(np.linspace(lims_k[0], lims_k[1], 5))
ax.set_xlabel(r"$t\ (s)$")
ax.set_ylabel(r"$\theta$")

legend = ax.legend(title=r"$h\ (s)$", bbox_to_anchor=(1.05, 1.0), loc="upper left")

plt.savefig("pendulum-paths.pdf")
plt.show()

In [ ]:
def render_animation(*, t, theta):
    h = t[1] - t[0]
    fps = min(50, int(1 / h))
    frames = int(T * fps)

    fig, ax = plt.subplots(figsize=(3, 3), layout="tight")

    x_pivot = pivot(t=t[0])
    y_pivot = 0.0
    x_mass = x_pivot + l * np.sin(theta[0])
    y_mass = y_pivot - l * np.cos(theta[0])

    scat = ax.scatter(x_pivot, y_pivot, c="k", marker="o")
    line = ax.plot([x_pivot, x_mass], [y_pivot, y_mass], c="k")[0]
    patch = ax.add_patch(Circle([x_mass, y_mass], 0.15, ec="k", fc="0.5", zorder=2))

    ax.set_xlim([-2.0, 2.0])
    ax.set_ylim([-2.0, 2.0])
    ax.set_xticks([-2, -1, 0, 1, 2])
    ax.set_yticks([-2, -1, 0, 1, 2])
    ax.set_aspect("equal")

    def update(frame):
        idx = np.sum(t < frame / fps)
        x_pivot = pivot(t=t[idx])
        y_pivot = 0.0
        x_mass = x_pivot + l * np.sin(theta[idx])
        y_mass = y_pivot - l * np.cos(theta[idx])

        scat.set_offsets([pivot(t=t[idx]), 0.0])
        line.set_xdata([x_pivot, x_mass])
        line.set_ydata([y_pivot, y_mass])
        patch.center = [x_mass, y_mass]

        return scat, line, patch

    ani = animation.FuncAnimation(
        fig=fig, func=update, frames=frames, interval=1000.0 / fps
    )
    fname = "pendulum_a-{:.3f}_omega-{:.3f}_h-{:.3f}.mp4".format(a, omega, h)
    fname = os.path.join("ani", fname)
    ani.save(filename=fname, writer="ffmpeg", dpi=300)

    plt.show()

In [ ]:
for t, theta in zip(ts, thetas):
    render_animation(t=t, theta=theta)